In [1]:
!pip uninstall -y -q torchao
!pip install -q --upgrade gradio

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 28.3 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:

"""
Ateliê Generativo — Pipeline Multimodal (Etapa 4)
Arquitetura Modernista de Brasília
"""

import os
import torch
import gradio as gr
from diffusers import StableDiffusionPipeline
from transformers import pipeline

# ----------------------------------------------------------------------
# Configuração — troque pelo seu repositório de LoRA no Hugging Face Hub
# ----------------------------------------------------------------------
MODELO_BASE = "stable-diffusion-v1-5/stable-diffusion-v1-5"
LORA_REPO = "andre-lla/lora-brasilia-rank16"  # troque pelo modelo escolhido na comparação
TOKEN_ESTILO = "estilo_brasilia"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# ----------------------------------------------------------------------
# Carregamento dos modelos (uma vez, na inicialização do Space)
# ----------------------------------------------------------------------
print("Carregando pipeline de difusão + LoRA...")
pipe_imagem = StableDiffusionPipeline.from_pretrained(
    MODELO_BASE, torch_dtype=DTYPE, safety_checker=None
).to(DEVICE)
pipe_imagem.load_lora_weights(LORA_REPO)

print("Carregando LLM")
expansor = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

print("Carregando texto-para-fala...")
tts = pipeline("text-to-speech", model="suno/bark-small", device_map="auto")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Carregando pipeline de difusão + LoRA...


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/353 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

speaker_embeddings_path.json:   0%|          | 0.00/61.1k [00:00<?, ?B/s]

In [3]:
# ----------------------------------------------------------------------
# Função principal do pipeline
# ----------------------------------------------------------------------
def gerar(tema):
    if not tema or not tema.strip():
        return "Digite um tema para começar.", None, None

    # 1. LLM expande o tema numa descrição visual rica, já com o token de estilo
    instrucao = (
        f"Expanda o tema '{tema}' em uma descrição visual rica e curta (uma frase), "
        f"de uma cena no estilo de arquitetura modernista de Brasília, "
        f"iniciando obrigatoriamente com '{TOKEN_ESTILO},'"
    )
    saida_llm = expansor(instrucao, max_new_tokens=60, do_sample=True, temperature=0.8)
    prompt = saida_llm[0]["generated_text"].strip()

    # garante que o token de estilo está presente, mesmo se o LLM "esquecer"
    if TOKEN_ESTILO not in prompt:
        prompt = f"{TOKEN_ESTILO}, {prompt}"

    # 2. Stable Diffusion + LoRA gera a imagem
    imagem = pipe_imagem(
        prompt,
        negative_prompt="desfocado, deformado, baixa qualidade",
        guidance_scale=7.5,
        num_inference_steps=30,
    ).images[0]

    # 3. TTS narra a descrição gerada
    audio_saida = tts(prompt)

    return prompt, imagem, (audio_saida["sampling_rate"], audio_saida["audio"])


# ----------------------------------------------------------------------
# Interface Gradio
# ----------------------------------------------------------------------
demo = gr.Interface(
    fn=gerar,
    inputs=gr.Textbox(
        label="Tema",
        placeholder="ex.: feira de domingo, pôr do sol, dia de eleição...",
    ),
    outputs=[
        gr.Textbox(label="Prompt expandido pelo LLM"),
        gr.Image(label="Imagem gerada (estilo: arquitetura modernista de Brasília)"),
        gr.Audio(label="Narração em áudio"),
    ],
    title="Ateliê Generativo — Arquitetura Modernista de Brasília",
    description=(
        "Digite um tema curto e veja-o reimaginado no estilo modernista de Brasília: "
        "curvas de concreto branco, céu do cerrado, monumentos icônicos. "
        "Projeto acadêmico — UniCEUB, disciplina de IA Generativa e Modelos Multimodais."
    ),
    examples=[
        ["feira de domingo"],
        ["pôr do sol na cidade"],
        ["um dia de eleição"],
        ["uma manhã de trabalho"],
    ],
)

if __name__ == "__main__":
    demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f1dd4ea710b6c79a68.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
